In [59]:
from evolutionSimulation.python.neuralnetworks.nn import * 
from datasets import load_dataset
import torch
import transformers

brain = Brain()
data = load_dataset("json", data_files=r"C:\Users\allan\nvim\evolutionSimulation\evolutionSimulation\python\dataset\simple_dataset.json")
shuffled_dataset = data.shuffle()

In [138]:
import torch
import time
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim


animals = {
    "lion": 1,
    "crocodile": 1,
    "dragon": 1,
    "duck": 0,
    "sheep": 0,
}

def batch(batch_size, start_index, dataset):
    truth = []
    images = [sample for sample in dataset["train"][start_index:start_index + batch_size]["image"]]
    tensor = torch.tensor(images)
    tensor = tensor.view(10, 1, 28, 28)
    for animal in dataset["train"][start_index:start_index + batch_size]["name"]:
        truth.append(animals[animal])
    return tensor.float(), truth

def train(num_img, batch_size, num_epoch, model, dataset):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = optim.Adam(model.parameters(), lr=1e-5)
    model.to(device)
    model.train()
    best_loss = float("inf")
    total_loss = 0

    for epoch in range(num_epoch):
        epoch_loss = 0
        t0 = time.perf_counter()

        for i in range(0, num_img, batch_size):
            temp_batch = batch(batch_size, i, dataset)
            predictions = model(temp_batch[0].to(device))
            ground_truth = torch.tensor(temp_batch[1]).to(device, dtype = torch.long)
            loss_fn = nn.CrossEntropyLoss()

            loss = loss_fn(predictions, ground_truth)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            if (i / num_img * 100) % 10 == 0:
                print(f"{i / num_img * 100}% | Loss: {loss.item():.4f}")
        
        avg_loss = epoch_loss / (num_img // batch_size)
        total_loss += epoch_loss
        t1 = time.perf_counter()
        print(f"Finished Epoch {epoch} in {t1 - t0} seconds, Loss: {avg_loss:.4f}")
        torch.save(model.state_dict(), r'C:\Users\allan\nvim\evolutionSimulation\evolutionSimulation\model_weights\model{}.pt'.format(epoch))


In [140]:
train(10000, 10, 3,  brain, shuffled_dataset)

0.0% | Loss: 22.1651
10.0% | Loss: 5.6174
20.0% | Loss: 5.5053
30.0% | Loss: 3.3171
40.0% | Loss: 2.5428
50.0% | Loss: 0.2068
60.0% | Loss: 1.7471
70.0% | Loss: 1.3376
80.0% | Loss: 1.3366
90.0% | Loss: 1.4887
Finished Epoch 0 in 10.189340499928221 seconds, Loss: 3.3991
0.0% | Loss: 0.9326
10.0% | Loss: 2.3888
20.0% | Loss: 1.3680
30.0% | Loss: 0.6636
40.0% | Loss: 1.0505
50.0% | Loss: 0.0340
60.0% | Loss: 0.7963
70.0% | Loss: 0.6372
80.0% | Loss: 0.9651
90.0% | Loss: 0.5129
Finished Epoch 1 in 10.357770000002347 seconds, Loss: 0.8864
0.0% | Loss: 0.1815
10.0% | Loss: 1.5303
20.0% | Loss: 0.5548
30.0% | Loss: 0.4718
40.0% | Loss: 0.7977
50.0% | Loss: 0.0207
60.0% | Loss: 0.4293
70.0% | Loss: 0.5837
80.0% | Loss: 0.9078
90.0% | Loss: 0.3470
Finished Epoch 2 in 9.664809299982153 seconds, Loss: 0.6416


In [142]:
brain = Brain()
brain.load_state_dict(torch.load(r'C:\Users\allan\nvim\evolutionSimulation\evolutionSimulation\model_weights\model2.pt'))

<All keys matched successfully>

In [157]:
import numpy as np
test2 = np.array(shuffled_dataset["train"][200002]["image"], dtype=np.uint8 )
print(shuffled_dataset["train"][10003]["name"])

sheep


In [156]:
brain(torch.tensor(shuffled_dataset["train"][10003]["image"]).view(1, 1, 28, 28).float())

tensor([[-15.4891, -20.4482]], grad_fn=<AddmmBackward0>)